## **🔧 1️⃣ Install and Initialize Earth Engine**

In [ ]:
# =====================================
# 1️⃣ Install and Initialize Earth Engine + Geemap
# =====================================
# !pip install earthengine-api geemap folium geopandas
import ee
import geemap
from IPython.display import display
import folium
import geemap.foliumap as geemap
from google.colab import files
import geopandas as gpd
import io
import pprint

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-eesaleemkhan')

### **Load shapefile from GEE Assets**

In [ ]:
# =====================================
# 2️⃣ Load AOI
# =====================================
aoi = ee.FeatureCollection("projects/ee-eesaleemkhan/assets/MeghalayDistt")
print(aoi.size().getInfo(), "polygons loaded")

Map = geemap.Map(center=[25.0, 91.0], zoom=7)
Map.setOptions('HYBRID')

aoi_style = {'color': 'red', 'fillColor': '00000000', 'width': 2}
Map.addLayer(aoi.style(**aoi_style), {}, "AOI")

# =====================================
# 3️⃣ Define Bands and Cloud/Snow Mask
# =====================================
bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']

def mask_s2_clouds(image):
    scl = image.select('SCL')
    # Mask cloud/shadow/cirrus/snow
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask).select(bands).divide(10000).copyProperties(image, ["system:time_start"])

# =====================================
# 4️⃣ Load Sentinel-2 Collection
# =====================================
start_date = '2025-10-01'
end_date = '2025-11-20'

s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5))
                 .map(mask_s2_clouds))

print("Number of images in collection:", s2_collection.size().getInfo())

# =====================================
# 5️⃣ Create Median Composite
# =====================================
s2_median = s2_collection.median().clip(aoi)

# =====================================
# 6️⃣ Compute Indices: NDVI, NDWI, NDBI
# =====================================
def add_indices(image):
    ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')
    ndbi = image.normalizedDifference(['B11','B8']).rename('NDBI')
    return image.addBands([ndvi, ndwi, ndbi])

s2_indexed = add_indices(s2_median)

# =====================================
# 7️⃣ Visualization: RGB + False-color + NDVI
# =====================================
# RGB
Map.addLayer(s2_indexed, {'bands':['B4','B3','B2'],'min':0,'max':0.3}, 'RGB Composite')
# False-color NIR
Map.addLayer(s2_indexed, {'bands':['B8','B4','B3'],'min':0,'max':0.3}, 'False-color NIR')
# NDVI
Map.addLayer(s2_indexed, {'bands':['NDVI'],'min':0,'max':1,'palette':['white','green']}, 'NDVI')
Map  # Last line to render Map


12 polygons loaded
Number of images in collection: 78


##**🧭 Export Composites Image**

In [ ]:
# =====================================
# 8️⃣ Export Composite Image (Auto-named)
# =====================================

# Import datetime to auto-generate month-year tags
import datetime

# Extract year and month from your start_date
start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d")
month_name = start_dt.strftime("%b")
year_str = start_dt.strftime("%Y")

# Create dynamic export name
export_name = f"S2_MSIL2_{year_str}_{month_name}"

# Select image to export (includes NDVI, NDWI, NDBI)
export_image = s2_indexed

# Define export region and parameters
export_region = aoi.geometry()
export_scale = 10

# ✅ Export to Earth Engine Asset
asset_id = f"projects/ee-eesaleemkhan/assets/{export_name}"
task_asset = ee.batch.Export.image.toAsset(
    image=export_image,
    description=f"Sub_{export_name}_toAsset",
    assetId=asset_id,
    region=export_region,
    scale=export_scale,
    maxPixels=1e13
)
task_asset.start()
print(f"🟢 Export to Asset started: {asset_id}")

# ✅ Export to Google Drive
task_drive = ee.batch.Export.image.toDrive(
    image=export_image,
    description=f"Sub_{export_name}_toDrive",
    folder='GEE_Exports',
    fileNamePrefix=export_name,
    region=export_region,
    scale=export_scale,
    maxPixels=1e13
)
task_drive.start()
print(f"🟡 Export to Google Drive started: {export_name}. Check your GEE Tasks tab.")


🟢 Export to Asset started: projects/ee-eesaleemkhan/assets/S2_MSIL2_2025_Oct
🟡 Export to Google Drive started: S2_MSIL2_2025_Oct. Check your GEE Tasks tab.


In [ ]:
# =====================================
# 9️⃣ Load and Clean Training Samples
# =====================================
training_samples = ee.FeatureCollection("projects/ee-eesaleemkhan/assets/Train_S2MSI_2022")

# Convert CLASS_ID from string to number
training_samples = training_samples.map(lambda f: f.set('CLASS_ID', ee.Number.parse(f.get('CLASS_ID'))))
print("Training samples:", training_samples.size().getInfo())

# =====================================
# 🔟 Split Data for Accuracy Assessment
#  Sample Composite using Training Data
# =====================================
training = s2_indexed.sampleRegions(
    collection=training_samples,
    properties=['CLASS_ID'],
    scale=20
)

Training samples: 1033


In [ ]:
# =====================================
# 1️⃣1️⃣ Split Data for Accuracy Assessment
# =====================================
# Random split (70% train, 30% test)
sample_with_random = training.randomColumn('random')
training_partition = sample_with_random.filter(ee.Filter.lt('random', 0.7))
validation_partition = sample_with_random.filter(ee.Filter.gte('random', 0.7))

print("Training samples:", training_partition.size().getInfo())
print("Validation samples:", validation_partition.size().getInfo())

Training samples: 290757
Validation samples: 123919


In [ ]:
# =====================================
# 1️⃣2️⃣ Train Random Forest Classifier
# =====================================
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    features=training_partition,
    classProperty='CLASS_ID',
    inputProperties=s2_median.bandNames()
)

# =====================================
# 1️⃣3️⃣ Classify the Composite
# =====================================
classified = s2_median.classify(classifier)

# =====================================
# ✅ Define CLASS_ID → Color & Label Mapping
# =====================================
class_info = {
    6: {'name': 'Water', 'color': '0000FF'},
    5: {'name': 'Tree_Covers', 'color': '008000'},
    4: {'name': 'Shrub_Grasslands', 'color': 'ADFF2F'},
    3: {'name': 'Croplands', 'color': 'FFFF00'},
    2: {'name': 'Builtup', 'color': 'FF0000'},
    1: {'name': 'Bare_Soil', 'color': '808080'}
}

# Ordered palette & labels
palette_ordered = [class_info[i]['color'] for i in sorted(class_info.keys())]
labels_ordered = [class_info[i]['name'] for i in sorted(class_info.keys())]

# =====================================
# 1️⃣4️⃣ Display Classification Map
# =====================================
Map = geemap.Map(center=[25.0, 91.0], zoom=7)
Map.setOptions('HYBRID')
# RGB Composite
Map.addLayer(s2_indexed, {'bands': ['B4','B3','B2'], 'min': 0, 'max': 0.3}, 'RGB Composite')
# False-color NIR
Map.addLayer(s2_indexed, {'bands': ['B8','B4','B3'], 'min': 0, 'max': 0.3}, 'False-color NIR')
# NDVI
Map.addLayer(s2_indexed.select('NDVI'), {'min':0, 'max':1, 'palette':['white','green']}, 'NDVI')
# Classification Map
Map.addLayer(
    classified,
    {'min': 1, 'max': 6, 'palette': ['#'+c for c in palette_ordered]},
    'Classification Map'
)

# 🧭 Add Legend
legend_dict = {v['name']: '#' + v['color'] for k,v in class_info.items()}
Map.add_legend(title="Land Cover Classes", legend_dict=legend_dict)

# Render the map
Map


In [ ]:
# =====================================
# 1️⃣5️⃣ Accuracy Assessment
# =====================================
# Apply classifier to validation samples
validated = validation_partition.classify(classifier)

# Confusion matrix
conf_matrix = validated.errorMatrix('CLASS_ID', 'classification')
print("Confusion Matrix:\n", conf_matrix.getInfo())

# Accuracy metrics
overall_acc = conf_matrix.accuracy()
kappa = conf_matrix.kappa()
producers_acc = conf_matrix.producersAccuracy()
users_acc = conf_matrix.consumersAccuracy()

print("Overall Accuracy:", overall_acc.getInfo())
print("Kappa Coefficient:", kappa.getInfo())
print("Producer’s Accuracy (per class):", producers_acc.getInfo())
print("User’s Accuracy (per class):", users_acc.getInfo())

Confusion Matrix:
 [[0, 0, 0, 0, 0, 0, 0], [0, 6519, 29, 109, 12, 4, 0], [0, 7, 453, 47, 1, 0, 1], [0, 89, 30, 15356, 59, 187, 1], [0, 16, 0, 45, 2169, 86, 0], [0, 0, 0, 66, 67, 94240, 1], [0, 0, 0, 10, 1, 3, 4311]]
Overall Accuracy: 0.9930115640055197
Kappa Coefficient: 0.9824867880446588
Producer’s Accuracy (per class): [[0], [0.9781207852540087], [0.888015717092338], [0.9772293601322987], [0.9339378238341969], [0.9985483289889164], [0.9967630057803468]]
User’s Accuracy (per class): [[0, 0.9840096545481973, 0.8895463510848126, 0.9826637666325486, 0.9376623376623376, 0.9969849884160081, 0.9993045897079277]]


In [ ]:
# =====================================
# 1️⃣6️⃣ Export Classified Map to Google Drive
# =====================================
export_task = ee.batch.Export.image.toDrive(
    image=classified,
    description='S2_Classification_March2022',
    folder='Sentinel2_Classification',        # Google Drive folder
    fileNamePrefix='S2_Classification_March2022',
    region=aoi.geometry(),                    # AOI geometry
    scale=10,                                 # Sentinel-2 resolution
    crs='EPSG:32646',                         # UTM Zone 46N
    maxPixels=1e13,                           # increase maxPixels for large AOI
    fileFormat='GeoTIFF'
)

# Start the export
export_task.start()
print("✅ Export started. Check your Google Drive folder 'Sentinel2_Classification'")


✅ Export started. Check your Google Drive folder 'Sentinel2_Classification'
